# 13 — Tier 5: intervention ranking and scenario simulation (Experiment 6)

Completes the last unmeasured experiment. Two objectives are implemented here:

* **(vi) Intervention ranking** — perturb each controllable cost parameter and
  *re-execute* the projection, reporting the measured displacement of the
  exhaustion date in weeks.
* **(vii) Scenario simulation** — run the identical path on a user-supplied
  modified parameter vector and return the differential.

**Experiment 6 as specified in the IDF** compares full-pipeline re-execution
against a linearised analytic approximation, "to quantify what the linearisation
would have discarded." That comparison is the substance of this notebook.

**Two scope limits, stated before the numbers.** First, the Tier 3 survival
estimator is fitted on funding history and sector — none of which a founder can
change — so perturbing a cost line moves the cash-flow arm and leaves the hazard
unchanged. Displacement here is displacement of the *projected exhaustion date*.
Second, the cost-line shares in `src/tier5.py` are a stated assumption, not
measurement: the snapshot carries no itemised expenses. Both inherit into every
figure below.

Reads: `data/processed/synthetic_trajectories.csv`
Writes: `reports/experiment6_intervention_ranking.csv`, `data/processed/intervention_rankings.csv`

In [1]:
import pandas as pd, numpy as np, sys, os
sys.path.insert(0, "../src")
import tier5

PROCESSED="../data/processed"; REPORTS="../reports"
os.makedirs(REPORTS, exist_ok=True)

traj = pd.read_csv(f"{PROCESSED}/synthetic_trajectories.csv")
print(f"[load] {len(traj):,} monthly values across {traj['object_id'].nunique():,} companies")
print(f"[cost lines] {tier5.COST_LINES}")

ModuleNotFoundError: No module named 'tier5'

## Build a company state for every venture with a trajectory

In [ ]:
CASH_MULTIPLE = 8.0   # placeholder for cash on hand; see tier5.state_from_trajectory

states = {}
for oid, g in traj.sort_values("month_idx").groupby("object_id"):
    spend = g["synthesized_spend"].values
    if len(spend) >= 3:
        states[oid] = tier5.state_from_trajectory(oid, spend, cash_multiple=CASH_MULTIPLE)

print(f"[states] built {len(states):,} company states (>=3 months of trajectory)")
base_runways = np.array([tier5.runway_months(s) for s in states.values()])
print(f"[runway] median baseline {np.median(base_runways):.1f} months, "
      f"IQR {np.percentile(base_runways,25):.1f}-{np.percentile(base_runways,75):.1f}")

## Objective (vi) — rank interventions for one venture

In [ ]:
demo_id = list(states)[0]
demo = states[demo_id]
print(f"Company {demo_id}   baseline runway {tier5.runway_months(demo):.2f} months\n")
print(f"{'intervention':38s} {'re-executed':>12s} {'analytic':>10s} {'error':>8s}")
for r in tier5.rank_interventions(demo, pct=0.20):
    analytic_w = tier5.analytic_gain_months(demo, r["cost_line"], 0.20) * 4.345
    print(f"{r['intervention']:38s} {r['gain_weeks']:9.2f} wk {analytic_w:8.2f} wk "
          f"{analytic_w - r['gain_weeks']:+7.2f}")

## Experiment 6 — how much does the linearisation discard?

Run the comparison across a large sample of companies and every cost line. The
question is whether the flat-burn identity `runway = cash / burn` is an adequate
substitute for re-executing the projection, or whether it systematically
misstates the gain.

In [ ]:
from scipy.stats import spearmanr

SAMPLE = min(3000, len(states))
rng = np.random.default_rng(42)
sample_ids = rng.choice(list(states), size=SAMPLE, replace=False)

rows=[]
for i, oid in enumerate(sample_ids, 1):
    s = states[oid]
    ranked = tier5.rank_interventions(s, pct=0.20)
    for r in ranked:
        analytic = tier5.analytic_gain_months(s, r["cost_line"], 0.20)
        if not np.isfinite(analytic):
            continue
        rows.append({"object_id": oid, "cost_line": r["cost_line"],
                     "reexecuted_months": r["gain_months"],
                     "analytic_months": analytic,
                     "reexecuted_weeks": r["gain_months"]*4.345,
                     "analytic_weeks": analytic*4.345})
    if i % 1000 == 0:
        print(f"   ... {i:,} / {SAMPLE:,} companies")

cmp = pd.DataFrame(rows)
cmp["abs_error_weeks"] = (cmp["analytic_weeks"] - cmp["reexecuted_weeks"]).abs()
cmp["signed_error_weeks"] = cmp["analytic_weeks"] - cmp["reexecuted_weeks"]
print(f"\n[Experiment 6] {len(cmp):,} intervention evaluations across {SAMPLE:,} companies")

In [ ]:
rho, p = spearmanr(cmp["reexecuted_weeks"], cmp["analytic_weeks"])
mae  = cmp["abs_error_weeks"].mean()
med  = cmp["abs_error_weeks"].median()
bias = cmp["signed_error_weeks"].mean()
overest = (cmp["signed_error_weeks"] > 0).mean()*100

print(f"[Experiment 6] rank correlation (Spearman rho) : {rho:.4f}")
print(f"[Experiment 6] mean absolute error             : {mae:.2f} weeks")
print(f"[Experiment 6] median absolute error           : {med:.2f} weeks")
print(f"[Experiment 6] mean signed error (bias)        : {bias:+.2f} weeks")
print(f"[Experiment 6] analytic overestimates in       : {overest:.1f}% of cases")
p90 = cmp["abs_error_weeks"].quantile(0.90)
p99 = cmp["abs_error_weeks"].quantile(0.99)
capped = (cmp["reexecuted_months"] > 100).mean()*100
print(f"[Experiment 6] 90th / 99th pct absolute error  : {p90:.2f} / {p99:.2f} weeks")
print(f"[Experiment 6] evaluations with >100mo gain     : {capped:.2f}%  (trend-driven, see note)")
print()
print("READ BOTH THE MEAN AND THE MEDIAN. They differ by more than an order of")
print("magnitude here, so the mean alone would misrepresent the typical case.")
print("The distribution is heavy-tailed for a substantive reason: where the fitted")
print("burn trend is flat or declining, cutting a cost line can push the zero-crossing")
print("out very far, and the flat-burn identity -- which has no notion of trend --")
print("cannot represent that at all. For the median venture the two methods agree")
print("closely; for the minority with a favourable trend they diverge enormously.")
print()
print("Interpretation: a high rank correlation with a large magnitude error means the")
print("linearisation orders the interventions correctly but misstates the size of the")
print("gain. Ordering is what a ranking needs; the magnitude is what a founder acts")
print("on, which is why the IDF specifies re-execution rather than a derivative.")

In [ ]:
by_line = cmp.groupby("cost_line").agg(
    n=("reexecuted_weeks","size"),
    mean_reexecuted_wk=("reexecuted_weeks","mean"),
    mean_analytic_wk=("analytic_weeks","mean"),
    mean_abs_error_wk=("abs_error_weeks","mean"),
).sort_values("mean_reexecuted_wk", ascending=False).round(3)
print(by_line.to_string())

summary = pd.DataFrame([{
    "companies_evaluated": SAMPLE,
    "intervention_evaluations": len(cmp),
    "spearman_rho": round(rho,4),
    "mean_abs_error_weeks": round(mae,3),
    "median_abs_error_weeks": round(med,3),
    "mean_signed_error_weeks": round(bias,3),
    "p90_abs_error_weeks": round(p90,3),
    "p99_abs_error_weeks": round(p99,3),
    "pct_analytic_overestimates": round(overest,1),
}])
summary.to_csv(f"{REPORTS}/experiment6_intervention_ranking.csv", index=False)
by_line.to_csv(f"{REPORTS}/experiment6_by_cost_line.csv")
print(f"\n[saved] {REPORTS}/experiment6_intervention_ranking.csv")

## Objective (vii) — scenario simulation

In [ ]:
scenarios = [
    ("Cut burn 20%",                dict(burn_multiplier=0.80)),
    ("Cut burn 30%",                dict(burn_multiplier=0.70)),
    ("Raise $2M",                   dict(new_raise=2_000_000)),
    ("Cut burn 20% and raise $2M",  dict(burn_multiplier=0.80, new_raise=2_000_000)),
    ("Add $50k/mo revenue",         dict(revenue_offset=50_000)),
]
print(f"Company {demo_id}   baseline {tier5.runway_months(demo):.2f} months\n")
print(f"{'scenario':32s} {'runway':>10s} {'delta':>10s}")
for name, kw in scenarios:
    out = tier5.simulate_scenario(demo, **kw)
    print(f"{name:32s} {out['scenario_months']:7.2f} mo {out['delta_months']:+7.2f} mo")

In [ ]:
# persist per-company rankings for the dashboard to read
out_rows=[]
for oid in sample_ids[:2000]:
    for r in tier5.rank_interventions(states[oid], pct=0.20):
        out_rows.append({"object_id": oid, **r})
pd.DataFrame(out_rows).to_csv(f"{PROCESSED}/intervention_rankings.csv", index=False)
print(f"[done] wrote {PROCESSED}/intervention_rankings.csv "
      f"({len(out_rows):,} rows, {min(2000,len(sample_ids)):,} companies)")